# EDA Notebook 5 — SarcOji Auxiliary Scope Gate

**Thesis:** *An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns*  
**Student:** Abhishek Tomar | PN1196973 | LJMU

## Purpose
SarcOji is **not** an approved fourth core sentiment dataset. This notebook decides whether it is sufficiently reproducible and relevant to retain only as an **auxiliary validation / limitation dataset for the frozen sarcasm-indicated tagging proxy within emoji-containing social-media text**.

It must **never** be used to:
- train, tune, select or rank the sentiment models;
- change the TweetEval subgroup rules or thresholds;
- alter the sarcasm regex after seeing SarcOji labels;
- manufacture subgroup sample size for TweetEval;
- claim general sarcasm-detection performance outside SarcOji's emoji-containing corpus.

The decision is based on provenance, reproducibility, data integrity, compatibility with the frozen rules, and interpretability — **not on obtaining a high F1 score**.

## Cell 1 — Install/import libraries and mount Drive

In [1]:
!pip install -q emoji pyarrow scikit-learn

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import glob, json, re, pickle, requests, hashlib
import numpy as np
import pandas as pd
import emoji

from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    balanced_accuracy_score, accuracy_score
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
print('Libraries loaded.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 3.0 MB/s eta 0:00:00
Mounted at /content/drive
Libraries loaded.


## Cell 2 — Locate frozen TweetEval configuration and create output folder

In [2]:
CONFIG_NAME = 'tweeteval_freeze_config.json'
matches = glob.glob(f'/content/drive/MyDrive/**/{CONFIG_NAME}', recursive=True)
if not matches:
    raise FileNotFoundError(f'{CONFIG_NAME} not found. Freeze EDA_01 first.')

preferred = [p for p in matches if '/Results/TweetEval/' in p.replace('\\','/')]
config_path = Path(preferred[0] if preferred else matches[0])

with open(config_path, 'r', encoding='utf-8') as f:
    freeze = json.load(f)

assert freeze['emoji_heavy']['operator'] == '>' and float(freeze['emoji_heavy']['threshold']) == 0.05
assert freeze['slang_heavy']['operator'] == '>' and float(freeze['slang_heavy']['threshold']) == 0.10
assert int(freeze['slang_heavy']['lexicon_size']) == 51
assert int(freeze['reference_unmarked']['minimum_word_count']) == 8
assert freeze['priority_rule'] == ['sarcasm-indicated','emoji-heavy','slang-heavy','reference-unmarked','other']

RESULTS_ROOT = config_path.parent.parent
OUTDIR = RESULTS_ROOT / 'SarcOji'
OUTDIR.mkdir(parents=True, exist_ok=True)

print('Frozen rule source:', config_path)
print('Output directory:', OUTDIR)

Frozen rule source: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/TweetEval/tweeteval_freeze_config.json
Output directory: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/SarcOji


## Cell 3 — Download the official SarcOji file reproducibly

In [3]:
REPO_URL = 'https://github.com/VanditaGroverKapila/SarcOji'
RAW_URL = 'https://raw.githubusercontent.com/VanditaGroverKapila/SarcOji/main/SARCOJI.df'
LOCAL = Path('/content/SARCOJI.df')

r = requests.get(RAW_URL, timeout=120)
r.raise_for_status()
LOCAL.write_bytes(r.content)

sha256 = hashlib.sha256(r.content).hexdigest()

print('Downloaded bytes:', f'{len(r.content):,}')
print('SHA256:', sha256)
print('Source:', RAW_URL)

Downloaded bytes: 6,265,003
SHA256: acdc7b40db84ca7e076f32f06eb74d861c86d5e0ac7752db1b13d4df93d35d2e
Source: https://raw.githubusercontent.com/VanditaGroverKapila/SarcOji/main/SARCOJI.df


## Cell 4 — Load dataset with format checks

In [4]:
# The repository distributes a .df binary object. Try common pandas/pickle readers
# without modifying the source file.
load_errors = []
obj = None

try:
    obj = pd.read_pickle(LOCAL)
except Exception as e:
    load_errors.append(f'pd.read_pickle: {e}')

if obj is None:
    try:
        with open(LOCAL, 'rb') as f:
            obj = pickle.load(f)
    except Exception as e:
        load_errors.append(f'pickle.load: {e}')

if obj is None:
    raise RuntimeError('Could not load SARCOJI.df.\n' + '\n'.join(load_errors))

if isinstance(obj, pd.DataFrame):
    df = obj.copy()
else:
    try:
        df = pd.DataFrame(obj)
    except Exception as e:
        raise TypeError(f'Loaded object type {type(obj)} is not convertible to DataFrame: {e}')

print('Rows:', f'{len(df):,}')
print('Columns:', list(df.columns))
print('Dtypes:')
print(df.dtypes.to_string())

Rows: 29,377
Columns: ['Sarcastic', 'Text', 'Emojis', 'MaxEmoji', 'MaxEmojiNumOccurence', 'MaxEmojiPos', 'TextSWN', 'TextVader', 'TextTextBlob', 'EmojiSWN', 'EmojiVader', 'EmojiTextBlob', 'MEmojiWN', 'MEVader', 'METB', 'ESR']
Dtypes:
Sarcastic                 int64
Text                     object
Emojis                   object
MaxEmoji                 object
MaxEmojiNumOccurence      int64
MaxEmojiPos               int64
TextSWN                 float64
TextVader               float64
TextTextBlob            float64
EmojiSWN                float64
EmojiVader              float64
EmojiTextBlob           float64
MEmojiWN                float64
MEVader                 float64
METB                    float64
ESR                     float64


## Cell 5 — Resolve required schema without silently guessing

In [5]:
def resolve_col(candidates):
    lower = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

TEXT_COL = resolve_col(['Text','text'])
LABEL_COL = resolve_col(['Sarcastic','sarcastic','label','Label'])
EMOJIS_COL = resolve_col(['Emojis','emojis'])

if TEXT_COL is None or LABEL_COL is None:
    raise KeyError(
        f'Required text/label fields not found. Columns are: {list(df.columns)}'
    )

work = df.copy()
work['text_raw'] = work[TEXT_COL].fillna('').astype(str)

# SarcOji README specifies 0=non-sarcastic, 1=sarcastic.
labels_num = pd.to_numeric(work[LABEL_COL], errors='coerce')
if labels_num.isna().any() or not set(labels_num.unique()).issubset({0,1}):
    raise ValueError(
        f'Unexpected sarcasm labels: {work[LABEL_COL].value_counts(dropna=False).head(20).to_dict()}'
    )
work['sarcastic_gold'] = labels_num.astype(int)

print('Resolved text column :', TEXT_COL)
print('Resolved label column:', LABEL_COL)
print('Resolved emoji column:', EMOJIS_COL)
print('Gold label counts:')
print(work['sarcastic_gold'].value_counts().sort_index().to_string())

Resolved text column : Text
Resolved label column: Sarcastic
Resolved emoji column: Emojis
Gold label counts:
sarcastic_gold
0    17929
1    11448


## Cell 6 — Provenance/schema gate

In [6]:
expected_repo_rows = 29377
expected_sarcastic = 11448
expected_non_sarcastic = 17929

schema_gate = {
    'rows_match_repository_readme': int(len(work)) == expected_repo_rows,
    'sarcastic_count_matches_readme': int((work['sarcastic_gold']==1).sum()) == expected_sarcastic,
    'non_sarcastic_count_matches_readme': int((work['sarcastic_gold']==0).sum()) == expected_non_sarcastic,
    'text_field_available': TEXT_COL is not None,
    'binary_gold_label_available': LABEL_COL is not None,
}

print(json.dumps(schema_gate, indent=2))
if not all(schema_gate.values()):
    print('WARNING — one or more repository-schema expectations did not match; investigate before RETAIN decision.')

{
  "rows_match_repository_readme": true,
  "sarcastic_count_matches_readme": true,
  "non_sarcastic_count_matches_readme": true,
  "text_field_available": true,
  "binary_gold_label_available": true
}


## Cell 7 — Missing, empty, duplicate and conflicting-label audit

In [7]:
def normalise_text(t):
    t = str(t).strip().lower()
    return re.sub(r'\s+', ' ', t)

work['text_norm'] = work['text_raw'].map(normalise_text)

missing_text = int(work[TEXT_COL].isna().sum()) if hasattr(work[TEXT_COL], 'isna') else 0
empty_text = int((work['text_raw'].str.strip()=='').sum())

dup_audit = (work.groupby('text_norm', observed=True)
    .agg(
        row_count=('text_raw','size'),
        label_count=('sarcastic_gold','nunique'),
        labels=('sarcastic_gold', lambda x: '|'.join(map(str, sorted(set(x))))),
        example_text=('text_raw','first')
    )
    .reset_index())

dup_groups = dup_audit[dup_audit['row_count']>1].copy()
conflict_groups = dup_groups[dup_groups['label_count']>1].copy()

print('Missing text:', missing_text)
print('Empty text:', empty_text)
print('Unique normalized texts:', f"{work['text_norm'].nunique():,}")
print('Duplicate normalized-text groups:', f'{len(dup_groups):,}')
print('Extra duplicate rows:', f"{int((dup_groups['row_count']-1).sum()):,}")
print('Conflicting-label duplicate groups:', f'{len(conflict_groups):,}')

Missing text: 0
Empty text: 0
Unique normalized texts: 28,355
Duplicate normalized-text groups: 719
Extra duplicate rows: 1,022
Conflicting-label duplicate groups: 37


## Cell 8 — Direct corpus-property audit

In [8]:
work['word_count'] = work['text_raw'].map(lambda t: len(str(t).split()))
work['emoji_count_recomputed'] = work['text_raw'].map(lambda t: len(emoji.emoji_list(str(t))))
work['has_emoji_recomputed'] = work['emoji_count_recomputed'] > 0
work['has_hash'] = work['text_raw'].str.contains('#', regex=False)
work['has_at'] = work['text_raw'].str.contains('@', regex=False)
work['has_url_like'] = work['text_raw'].str.contains(r'https?://|www\.', regex=True, case=False)

corpus_audit = pd.DataFrame([
    ['rows', len(work), 100.0],
    ['rows_with_recomputed_emoji', int(work['has_emoji_recomputed'].sum()), float(work['has_emoji_recomputed'].mean()*100)],
    ['rows_with_hash_character', int(work['has_hash'].sum()), float(work['has_hash'].mean()*100)],
    ['rows_with_at_character', int(work['has_at'].sum()), float(work['has_at'].mean()*100)],
    ['rows_with_url_like_text', int(work['has_url_like'].sum()), float(work['has_url_like'].mean()*100)],
], columns=['measure','count','percentage'])

print(corpus_audit.to_string(index=False))
print('\nWord-count summary:')
print(work['word_count'].describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_string())
print('\nRecomputed emoji-count summary:')
print(work['emoji_count_recomputed'].describe(percentiles=[.25,.5,.75,.9,.95,.99]).to_string())

                   measure  count  percentage
                      rows  29377  100.000000
rows_with_recomputed_emoji  29377  100.000000
  rows_with_hash_character     85    0.289342
    rows_with_at_character    119    0.405079
   rows_with_url_like_text      1    0.003404

Word-count summary:
count    29377.000000
mean        14.254110
std         19.894837
min          1.000000
25%          7.000000
50%         11.000000
75%         18.000000
90%         24.000000
95%         29.000000
99%         57.000000
max        877.000000

Recomputed emoji-count summary:
count    29377.000000
mean         1.762161
std          2.422036
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
90%          3.000000
95%          4.000000
99%         10.000000
max         92.000000


## Cell 9 — Apply the frozen TweetEval linguistic rules unchanged

In [9]:
SLANG = list(freeze['slang_heavy']['lexicon'])
SLANG_TOKEN_RE = re.compile(
    r'(?<!\w)(' + '|'.join(sorted(map(re.escape, SLANG), key=len, reverse=True)) + r')(?!\w)',
    re.I
)
SARCASM_RE = re.compile(freeze['sarcasm_indicated']['regex'], re.I)

EMOJI_THRESHOLD = float(freeze['emoji_heavy']['threshold'])
SLANG_THRESHOLD = float(freeze['slang_heavy']['threshold'])
REF_MIN_WORDS = int(freeze['reference_unmarked']['minimum_word_count'])

def slang_count(t):
    return len(SLANG_TOKEN_RE.findall(str(t).lower()))

work['slang_count'] = work['text_raw'].map(slang_count)
denom = work['word_count'].clip(lower=1)
work['emoji_density'] = work['emoji_count_recomputed'] / denom
work['slang_ratio'] = work['slang_count'] / denom

work['is_emoji_heavy'] = work['emoji_density'] > EMOJI_THRESHOLD
work['is_slang_heavy'] = work['slang_ratio'] > SLANG_THRESHOLD
work['is_sarcasm_indicated'] = work['text_raw'].map(lambda t: bool(SARCASM_RE.search(str(t).lower())))
work['is_reference_candidate'] = (
    ~work['is_emoji_heavy'] &
    ~work['is_slang_heavy'] &
    ~work['is_sarcasm_indicated'] &
    (work['word_count'] >= REF_MIN_WORDS)
)

def assign_group(r):
    if r['is_sarcasm_indicated']: return 'sarcasm-indicated'
    if r['is_emoji_heavy']: return 'emoji-heavy'
    if r['is_slang_heavy']: return 'slang-heavy'
    if r['is_reference_candidate']: return 'reference-unmarked'
    return 'other'

work['subgroup'] = work.apply(assign_group, axis=1)

print('Frozen TweetEval rules applied unchanged.')
print(work['subgroup'].value_counts().to_string())

Frozen TweetEval rules applied unchanged.
subgroup
emoji-heavy           25978
reference-unmarked     3326
sarcasm-indicated        63
slang-heavy              10


## Cell 10 — Pre-priority overlap audit

In [10]:
focus = ['is_emoji_heavy','is_slang_heavy','is_sarcasm_indicated']
overlap = pd.DataFrame(index=focus, columns=focus, dtype=int)
for a in focus:
    for b in focus:
        overlap.loc[a,b] = int((work[a] & work[b]).sum())

flag_prev = pd.DataFrame({
    'flag': focus + ['is_reference_candidate'],
    'count': [int(work[c].sum()) for c in focus + ['is_reference_candidate']]
})
flag_prev['percentage'] = flag_prev['count']/len(work)*100

print(flag_prev.to_string(index=False))
print('\nOverlap matrix:')
print(overlap.to_string())

                  flag  count  percentage
        is_emoji_heavy  26024   88.586309
        is_slang_heavy    850    2.893420
  is_sarcasm_indicated     63    0.214453
is_reference_candidate   3326   11.321782

Overlap matrix:
                      is_emoji_heavy  is_slang_heavy  is_sarcasm_indicated
is_emoji_heavy               26024.0           840.0                  46.0
is_slang_heavy                 840.0           850.0                   0.0
is_sarcasm_indicated            46.0             0.0                  63.0


## Cell 11 — Validate the frozen sarcasm-indicated proxy against SarcOji gold labels

In [11]:
y_true = work['sarcastic_gold'].astype(int).to_numpy()
y_pred = work['is_sarcasm_indicated'].astype(int).to_numpy()

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
tn, fp, fn, tp = cm.ravel()

metrics = {
    'n': int(len(work)),
    'gold_sarcastic_n': int((y_true==1).sum()),
    'gold_non_sarcastic_n': int((y_true==0).sum()),
    'proxy_positive_n': int((y_pred==1).sum()),
    'proxy_negative_n': int((y_pred==0).sum()),
    'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
    'accuracy': float(accuracy_score(y_true,y_pred)),
    'precision': float(precision_score(y_true,y_pred,zero_division=0)),
    'recall_sensitivity': float(recall_score(y_true,y_pred,zero_division=0)),
    'specificity': float(tn/(tn+fp)) if (tn+fp) else np.nan,
    'f1': float(f1_score(y_true,y_pred,zero_division=0)),
    'balanced_accuracy': float(balanced_accuracy_score(y_true,y_pred)),
}

print('Confusion matrix [[TN, FP],[FN, TP]]:')
print(cm)
print()
for k,v in metrics.items():
    print(f'{k}: {v:.6f}' if isinstance(v,float) else f'{k}: {v}')

Confusion matrix [[TN, FP],[FN, TP]]:
[[17892    37]
 [11422    26]]

n: 29377
gold_sarcastic_n: 11448
gold_non_sarcastic_n: 17929
proxy_positive_n: 63
proxy_negative_n: 29314
tp: 26
fp: 37
tn: 17892
fn: 11422
accuracy: 0.609933
precision: 0.412698
recall_sensitivity: 0.002271
specificity: 0.997936
f1: 0.004517
balanced_accuracy: 0.500104


## Cell 12 — Interpretation of proxy validation

This is **not** a sarcasm model-development experiment.

A high or low score does not trigger rule changes. The frozen proxy is evaluated exactly as defined on TweetEval. SarcOji's gold label is used only to quantify how often that narrow surface-cue proxy overlaps with a labelled sarcasm corpus restricted to emoji-containing text.

Important interpretation limits:
- SarcOji contains emoji-bearing examples by design, so it cannot validate general social-media sarcasm prevalence.
- If explicit hashtags/mentions were removed upstream, a hashtag-based cue cannot be evaluated fully.
- Duplicate/conflicting texts are reported rather than silently removed for the primary scope decision.
- Proxy performance is descriptive evidence about the operational tag, not proof of authorial intent or a general sarcasm detector.

## Cell 13 — Emoji–sarcasm relationship

In [12]:
emoji_sarc = (work.groupby('sarcastic_gold', observed=True)
    .agg(
        n=('text_raw','size'),
        mean_emoji_count=('emoji_count_recomputed','mean'),
        median_emoji_count=('emoji_count_recomputed','median'),
        mean_emoji_density=('emoji_density','mean'),
        emoji_heavy_n=('is_emoji_heavy','sum'),
        sarcasm_proxy_n=('is_sarcasm_indicated','sum')
    )
    .reset_index())

emoji_sarc['emoji_heavy_pct'] = emoji_sarc['emoji_heavy_n']/emoji_sarc['n']*100
emoji_sarc['sarcasm_proxy_pct'] = emoji_sarc['sarcasm_proxy_n']/emoji_sarc['n']*100

print(emoji_sarc.to_string(index=False))

 sarcastic_gold     n  mean_emoji_count  median_emoji_count  mean_emoji_density  emoji_heavy_n  sarcasm_proxy_n  emoji_heavy_pct  sarcasm_proxy_pct
              0 17929          1.519996                 1.0            0.199281          15753               37        87.863238           0.206370
              1 11448          2.141422                 1.0            0.269102          10271               26        89.718728           0.227114


## Cell 14 — Scope-gate decision

In [13]:
# The RETAIN/DROP decision is based on whether the corpus is reproducible and
# interpretable for the narrow auxiliary purpose — never on achieving a high
# prediction score.
gate = {
    'official_repository_accessible': True,
    'source_file_downloaded_and_hashed': LOCAL.exists() and LOCAL.stat().st_size > 0,
    'repository_row_count_matches': bool(schema_gate['rows_match_repository_readme']),
    'repository_class_counts_match': bool(
        schema_gate['sarcastic_count_matches_readme'] and
        schema_gate['non_sarcastic_count_matches_readme']
    ),
    'full_text_available': bool((work['text_raw'].str.len() > 0).all()),
    'binary_sarcasm_labels_available': True,
    'frozen_tweeteval_rules_used_without_tuning': True,
    'not_used_for_sentiment_training_or_selection': True,
    'emoji_restricted_corpus_explicitly_acknowledged': True,
    'duplicates_and_conflicts_reported': True,
}

retain = all(gate.values())
decision = 'RETAIN' if retain else 'DROP'

print('Scope-gate checks:')
for k,v in gate.items():
    print(f' - {k}: {v}')
print()
print('FINAL SARCOJI SCOPE DECISION:', decision)
if retain:
    print('Permitted role: auxiliary validation/limitation evidence for the frozen sarcasm-indicated proxy within emoji-containing social-media text only.')
    print('Forbidden roles: sentiment training, model selection, subgroup threshold tuning, general sarcasm prevalence estimation, or fourth core dataset.')
else:
    print('Do not use SarcOji in the thesis empirical analysis unless the failed reproducibility/scope conditions are resolved.')

Scope-gate checks:
 - official_repository_accessible: True
 - source_file_downloaded_and_hashed: True
 - repository_row_count_matches: True
 - repository_class_counts_match: True
 - full_text_available: True
 - binary_sarcasm_labels_available: True
 - frozen_tweeteval_rules_used_without_tuning: True
 - not_used_for_sentiment_training_or_selection: True
 - emoji_restricted_corpus_explicitly_acknowledged: True
 - duplicates_and_conflicts_reported: True

FINAL SARCOJI SCOPE DECISION: RETAIN
Permitted role: auxiliary validation/limitation evidence for the frozen sarcasm-indicated proxy within emoji-containing social-media text only.
Forbidden roles: sentiment training, model selection, subgroup threshold tuning, general sarcasm prevalence estimation, or fourth core dataset.


## Cell 15 — Save authoritative SarcOji outputs

In [14]:
work.to_parquet(OUTDIR/'df_sarcoji_auxiliary_final.parquet', index=False)
dup_groups.to_csv(OUTDIR/'sarcoji_duplicate_groups.csv', index=False)
conflict_groups.to_csv(OUTDIR/'sarcoji_conflicting_label_duplicates.csv', index=False)
corpus_audit.to_csv(OUTDIR/'sarcoji_corpus_audit.csv', index=False)
flag_prev.to_csv(OUTDIR/'sarcoji_flag_prevalence.csv', index=False)
overlap.to_csv(OUTDIR/'sarcoji_overlap_matrix.csv')
emoji_sarc.to_csv(OUTDIR/'sarcoji_emoji_sarcasm_summary.csv', index=False)

with open(OUTDIR/'sarcoji_proxy_validation_metrics.json','w',encoding='utf-8') as f:
    json.dump(metrics,f,indent=2)

integrity = {
    'rows': int(len(work)),
    'unique_normalized_texts': int(work['text_norm'].nunique()),
    'duplicate_normalized_text_groups': int(len(dup_groups)),
    'extra_duplicate_rows': int((dup_groups['row_count']-1).sum()),
    'conflicting_label_duplicate_groups': int(len(conflict_groups)),
    'missing_text_rows': int(missing_text),
    'empty_text_rows': int(empty_text),
    'rows_with_recomputed_emoji': int(work['has_emoji_recomputed'].sum()),
    'sha256_source_file': sha256
}
with open(OUTDIR/'sarcoji_integrity_summary.json','w',encoding='utf-8') as f:
    json.dump(integrity,f,indent=2)

scope = {
    'dataset':'SarcOji',
    'repository':REPO_URL,
    'source_file':RAW_URL,
    'status':decision,
    'role_if_retained':'auxiliary validation/limitation evidence for frozen sarcasm-indicated tagging proxy within emoji-containing social-media text',
    'core_dataset':False,
    'used_for_sentiment_training':False,
    'used_for_model_selection':False,
    'used_for_feature_selection':False,
    'used_for_threshold_tuning':False,
    'rule_source':'TweetEval frozen configuration',
    'performance_based_rule_changes_permitted':False,
    'general_sarcasm_detector_claim_permitted':False,
    'general_sarcasm_prevalence_claim_permitted':False,
    'limitations':[
        'corpus restricted to emoji-containing text',
        'source corpora and preprocessing may alter availability of explicit surface cues',
        'proxy measures observable sarcasm indicators rather than verified authorial intent',
        'auxiliary analysis does not expand approved core sentiment dataset scope'
    ],
    'gate_checks':gate
}
with open(OUTDIR/'sarcoji_scope_decision.json','w',encoding='utf-8') as f:
    json.dump(scope,f,indent=2)

print('Saved SarcOji scope-gate outputs to:', OUTDIR)
for p in sorted(OUTDIR.iterdir()):
    print(' -',p.name)

Saved SarcOji scope-gate outputs to: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/SarcOji
 - df_sarcoji_auxiliary_final.parquet
 - sarcoji_conflicting_label_duplicates.csv
 - sarcoji_corpus_audit.csv
 - sarcoji_duplicate_groups.csv
 - sarcoji_emoji_sarcasm_summary.csv
 - sarcoji_flag_prevalence.csv
 - sarcoji_integrity_summary.json
 - sarcoji_overlap_matrix.csv
 - sarcoji_proxy_validation_metrics.json
 - sarcoji_scope_decision.json


## Cell 16 — Final report

In [15]:
print('='*76)
print('SARCOJI AUXILIARY SCOPE GATE')
print('='*76)
print('Rows:', f'{len(work):,}')
print('Gold labels:', work['sarcastic_gold'].value_counts().sort_index().to_dict())
print('Duplicate groups:', len(dup_groups))
print('Conflicting-label duplicate groups:', len(conflict_groups))
print('Rows with recomputed emoji:', int(work['has_emoji_recomputed'].sum()))
print('Frozen sarcasm-indicated proxy positives:', int(work['is_sarcasm_indicated'].sum()))
print('Proxy precision:', f"{metrics['precision']:.4f}")
print('Proxy recall:', f"{metrics['recall_sensitivity']:.4f}")
print('Proxy F1:', f"{metrics['f1']:.4f}")
print('Proxy balanced accuracy:', f"{metrics['balanced_accuracy']:.4f}")
print()
print('Decision:', decision)
print('No TweetEval rule or threshold has been changed.')

SARCOJI AUXILIARY SCOPE GATE
Rows: 29,377
Gold labels: {0: 17929, 1: 11448}
Duplicate groups: 719
Conflicting-label duplicate groups: 37
Rows with recomputed emoji: 29377
Frozen sarcasm-indicated proxy positives: 63
Proxy precision: 0.4127
Proxy recall: 0.0023
Proxy F1: 0.0045
Proxy balanced accuracy: 0.5001

Decision: RETAIN
No TweetEval rule or threshold has been changed.
